# GCC118 - Programação Matemática
## Universidade Federal de Lavras
### Instituto de Ciências Exatas e Tecnológicas
#### Lucio Vargas de Albuquerque Nunes

## Problema

Uma agroindústria deve produzir um tipo de ração para um determinado animal. Essa ração possui três ingredientes básicos: osso, soja e restos de peixe. Cada um desses três ingredientes contém diferentes quantidades de proteína e cálcio (para 1 Kg de ração).

Determinar em que quantidades devem ser misturados de modo a produzir uma ração que satisfaça às restrições nutricionais com custo mínimo.

### Dados

| Nutrientes        | Osso | Soja | Peixe | Ração |
| ----------------- | ---- | ---- | ----- | ----- |
| **Proteína**      | 0,2  | 0,5  | 0,4   | 0,3   |
| **Cálcio**        | 0,6  | 0,4  | 0,4   | 0,5   |
| **Custos ($/Kg)** | 0,56 | 0,81 | 0,46  | —     |


## Instalação da biblioteca PuLP

Para mais informações, acesse: [PuLP
](https://pypi.org/project/PuLP/).

In [1]:
!pip install pulp
import pulp


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Problema da Ração

O objetivo é determinar a proporção de cada ingrediente na composição da ração, de modo que o custo total seja minimizado e as exigências nutricionais de proteína e cálcio sejam satisfeitas.

### Variáveis de decisão

Definimos:

- $q_o$: quantidade de **osso** na ração;
- $q_s$: quantidade de **soja** na ração;
- $q_p$: quantidade de **peixe** na ração.

Como estamos considerando uma unidade de ração, as quantidades representam proporções e devem somar 1.

### Função objetivo

O custo total da ração é dado pela soma do custo de cada ingrediente multiplicado pela quantidade utilizada.

Assim, queremos minimizar:

$$
\min \quad q_o c_o + q_s c_s + q_p c_p
$$

Substituindo os custos fornecidos:

$$
\min \quad 0.56q_o + 0.81q_s + 0.46q_p
$$

### Restrições nutricionais

A ração deve possuir pelo menos **0,3 de proteína**.

Como osso, soja e peixe possuem, respectivamente, 0,2, 0,5 e 0,4 de proteína, temos:

$$
0.2q_o + 0.5q_s + 0.4q_p \geq 0.3
$$

Da mesma forma, a ração deve possuir pelo menos **0,5 de cálcio**.

Os conteúdos de cálcio dos ingredientes são 0,6 para osso, 0,4 para soja e 0,4 para peixe:

$$
0.6q_o + 0.4q_s + 0.4q_p \geq 0.5
$$

### Restrição de composição

A soma das proporções dos ingredientes deve ser igual a 1:

$$
q_o + q_s + q_p = 1
$$

Além disso, nenhuma quantidade pode ser negativa:

$$
q_o, q_s, q_p \geq 0
$$

### Modelo completo

Portanto, o problema de programação linear pode ser escrito como:

$$
\begin{aligned}
\min \quad & 0.56q_o + 0.81q_s + 0.46q_p \\[4pt]
\text{sujeito a} \quad
& 0.2q_o + 0.5q_s + 0.4q_p \geq 0.3 \\[2pt]
& 0.6q_o + 0.4q_s + 0.4q_p \geq 0.5 \\[2pt]
& q_o + q_s + q_p = 1 \\[2pt]
& q_o, q_s, q_p \geq 0
\end{aligned}
$$

In [2]:
custo = [0.56, 0.81, 0.46]

racao = [0.3, 0.5, 1]

nutrientes = [
    [0.2, 0.5, 0.4],  # proteína
    [0.6, 0.4, 0.4],  # cálcio
    [1.0, 1.0, 1.0]   # composição total
]

## Declaração do objeto que representa o modelo matemático

In [3]:
modelo = pulp.LpProblem("problema_racao", pulp.LpMinimize)

In [4]:
q_o = pulp.LpVariable("q_o", lowBound=0)
q_s = pulp.LpVariable("q_s", lowBound=0)
q_p = pulp.LpVariable("q_p", lowBound=0)

q_vars = [q_o, q_s, q_p]

In [5]:
modelo += pulp.lpSum(
    [custo[i] * q_vars[i] for i in range(3)]
)

In [6]:
# Proteína
modelo += (
    0.2 * q_o
    + 0.5 * q_s
    + 0.4 * q_p
    >= 0.3
)

# Cálcio
modelo += (
    0.6 * q_o
    + 0.4 * q_s
    + 0.4 * q_p
    >= 0.5
)

# A ração deve totalizar 1
modelo += (
    q_o + q_s + q_p == 1
)

### Resolvendo o problema

In [7]:
status = modelo.solve()

## Imprimindo as soluções do problema

In [8]:
print("status:", pulp.LpStatus[status])

print("funcao objetivo:", modelo.objective.value())

print("solucoes")

for variavel in q_vars:
    print(variavel.name, variavel.value())

status: Optimal
funcao objetivo: 0.51
solucoes
q_o 0.5
q_s 0.0
q_p 0.5


## Interpretação da solução

O modelo encontrou a seguinte solução ótima:

$$
q_o = 0.5
$$

$$
q_s = 0
$$

$$
q_p = 0.5
$$

Isso significa que a composição ótima da ração é:

- **50% de osso**;
- **0% de soja**;
- **50% de peixe**.

O custo mínimo obtido foi:

$$
Z^* = 0.51
$$

Portanto, para produzir uma unidade de ração, o menor custo possível é **0,51**.